# 6장 인터페이스 어댑터 계층: 컨트롤러와 프레젠터

파이썬으로 구현하는 클린 아키텍처 - 6장 인터페이스 어댑터 계층: 컨트롤러와 프레젠터 코드 예제

> **[노트북 참고]** 아래 셀은 노트북 환경에서 `TodoApp` 코드를 import할 수 있도록 경로를 설정합니다. 반드시 첫 번째로 실행해 주세요.

In [ ]:
# ============================================================
# [추가] 노트북 환경 설정
# TodoApp 패키지를 import하기 위한 경로 설정 (Colab/로컬 환경 자동 감지)
# 반드시 첫 번째로 실행해 주세요.
# ============================================================
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/songys/Clean-Architecture-with-Python.git /content/repo
    TODOAPP_PATH = '/content/repo/Chapter_6/TodoApp'
else:
    TODOAPP_PATH = os.path.join(os.getcwd(), 'TodoApp')

if TODOAPP_PATH not in sys.path:
    sys.path.insert(0, TODOAPP_PATH)

## 개요

이 장에서는 파이썬으로 인터페이스 어댑터 계층을 구현하는 방법을 살펴보고, 클린 아키텍처의 의존성 규칙을 어떻게 준수하는지 설명한다.컨트롤러가 외부 입력과 유스 케이스를 어떻게 연결하는지, 프레젠터가 다양한 출력 요구에 맞춰 도메인 데이터를 어떻게 변환하는지를 다룬다.

이 장에서 다루는 주요 주제:
* 인터페이스 어댑터 계층 설계
* 파이썬에서 컨트롤러 구현

### 00_task_controller.py

## 컨트롤러 패턴

인터페이스 어댑터 계층 구성 요소를 ﻿살펴본 것과 같이, 클린 아키텍처의 컨트롤러는 명확한 책임이 있다. 외부 소스에서 입력을 받고, 해당 입력을 검증 및 변환하여 유스 케이스가 기대하는 형식으로 만들고, 유스 케이스 실행을 조율하며, 적절히 결과를 처리한다.

> **[추가]** 아래 셀은 `TaskController` 코드 예제에서 사용하는 클래스들을 TodoApp에서 import합니다.

In [ ]:
# [추가] TaskController 예제에 필요한 import
# 유스케이스, DTO, 프레젠터, 뷰 모델 등 인터페이스 어댑터 계층의 핵심 구성 요소
from todo_app.application.use_cases.task_use_cases import CreateTaskUseCase
from todo_app.application.dtos.task_dtos import CreateTaskRequest, TaskResponse
from todo_app.interfaces.presenters.base import TaskPresenter
from todo_app.interfaces.view_models.base import OperationResult, ErrorViewModel
from todo_app.interfaces.view_models.task_vm import TaskViewModel

In [ ]:
# 컨트롤러 패턴 - 외부 입력을 유스케이스로 전달하고 결과를 뷰 모델로 변환하는 중재자
# 프레임워크 독립적 설계: Flask, Django 등 특정 웹 프레임워크에 의존하지 않는 구조
from dataclasses import dataclass


@dataclass
class TaskController:
    # 의존성 주입: 유스케이스와 프레젠터를 추상 인터페이스로 받아 느슨한 결합 유지
    create_use_case: CreateTaskUseCase
    # 필요에 따라 추가 유스 케이스 정의
    presenter: TaskPresenter

    # 작업 생성 요청 처리 - 5단계 데이터 변환 흐름
    def handle_create(self, title: str, description: str) -> OperationResult[TaskViewModel]:
        try:
            # 1단계: 외부 입력 → 요청 DTO로 변환 및 검증
            request = CreateTaskRequest(title=title, description=description)
            # 2단계: 유스케이스 실행 (도메인 로직 조율)
            result = self.create_use_case.execute(request)

            if result.is_success:
                # 3단계: 도메인 응답 → 뷰 모델로 변환 (프레젠터 활용)
                view_model = self.presenter.present_task(result.value)
                return OperationResult.succeed(view_model)

            # 4단계: 비즈니스 오류를 에러 뷰 모델로 변환
            error_vm = self.presenter.present_error(
                result.error.message, str(result.error.code.name)
            )
            return OperationResult.fail(error_vm.message, error_vm.code)

        except ValueError as e:
            # 5단계: 입력 검증 오류를 에러 뷰 모델로 변환
            error_vm = self.presenter.present_error(str(e), "VALIDATION_ERROR")
            return OperationResult.fail(error_vm.message, error_vm.code)

### 01_tightly_coupled_task_controller.py

## 안티패턴: 강하게 결합된 컨트롤러

이 컨트롤러는 클린 아키텍처의 몇 가지 핵심 원칙을 잘 보여 준다. 먼저, 주입된 의존성에만 의존한다는 점에 주목해 보자.

> **[추가]** 아래 안티패턴 예제에서 사용하는 `TaskUseCase`, `SqliteTaskRepository`, `CliTaskPresenter`는 실제 TodoApp에 없는 가상의 구현체입니다. 코드가 실행될 수 있도록 최소한의 스텁을 정의합니다.

In [ ]:
# [추가] 안티패턴 예제용 스텁 클래스 정의
# 아래 클래스들은 강한 결합의 문제점을 보여주기 위한 가상의 구현체

class TaskUseCase:
    """[스텁] 안티패턴 예제용 가상 유스 케이스"""
    def __init__(self, repository=None):
        self.repository = repository
    def execute(self, request):
        pass

class SqliteTaskRepository:
    """[스텁] 안티패턴 예제용 가상 리포지토리 - 구체 구현체"""
    pass

class CliTaskPresenter:
    """[스텁] 안티패턴 예제용 가상 프레젠터 - 구체 구현체"""
    pass

In [ ]:
# 안티패턴: 강하게 결합된 컨트롤러 - 구체 구현체를 직접 생성
# → 의존성 주입 없이 내부에서 직접 인스턴스화하여 테스트와 교체가 어려운 구조
class TightlyCoupledTaskController:

    def __init__(self):

        # 구현체를 직접 생성하여 강한 결합 발생 (의존성 주입 미사용)
        # → SQLite 리포지토리와 CLI 프레젠터에 고정되어 교체 불가
        self.use_case = TaskUseCase(SqliteTaskRepository())

        self.presenter = CliTaskPresenter()

    def handle_create(self, title: str, description: str):

        # 구현 세부 사항 생략

        pass

### 02_create_task_request.py

## 작업 생성 요청 모델

외부 입력과 유스케이스 사이에 깔끔한 경계를 유지하는 요청 DTO이다. `to_execution_params()`로 API 형식을 도메인 타입으로 변환한다.

> **[추가]** 아래 `CreateTaskRequest` 예제에서 사용하는 `Priority`를 TodoApp에서 import합니다.

In [ ]:
# [추가] CreateTaskRequest 예제에 필요한 Priority import
from todo_app.domain.value_objects import Priority

In [ ]:
# 작업 생성 요청 DTO - 외부 입력과 유스케이스 사이의 경계 객체
# 입력 데이터 검증 + 도메인 타입 변환을 캡슐화하여 유스케이스의 순수성 보장
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class CreateTaskRequest:
    """새로운 작업(Task)을 생성하기 위한 요청 데이터"""

    title: str

    description: str

    due_date: Optional[str] = None  # API에서 문자열로 전달되는 마감일

    priority: Optional[str] = None  # API에서 문자열로 전달되는 우선순위

    # API 형식(문자열)을 도메인 타입으로 변환하는 경계 변환 메서드
    def to_execution_params(self) -> dict:
        """요청 데이터를 유스 케이스 실행에 필요한 파라미터로 변환"""

        params = {
            "title": self.title.strip(),
            "description": self.description.strip(),
        }

        # 문자열 → Priority 열거형으로 변환
        if self.priority:

            params["priority"] = Priority[self.priority.upper()]

        return params


"""
# TaskController에서의 사용 예시 - 요청 DTO가 경계에서 데이터를 검증하고 변환

try:

    request = CreateTaskRequest(title=title, description=description)

    # 요청이 검증되고 적절한 형식으로 변환됨

    result = self.create_use_case.execute(request)

except ValueError as e:

    # 유스 케이스에 도달하기 전에 검증 오류를 처리

    return OperationResult.fail(str(e), "VALIDATION_ERROR")

"""

### 03_task_contoller_independence.py

## 프레임워크 독립적 컨트롤러

`TaskController`는 추상화에만 의존하며, 웹 프레임워크 import·데이터베이스 코드가 없는 순수한 중재자이다. 파이썬에서는 복잡한 계약에 ABC를, 단순한 인터페이스에 덕 타이핑을 활용할 수 있다. 두 접근 방식 모두 클린 아키텍처의 의존성 원칙을 준수한다.

In [ ]:
# 프레임워크 독립적 컨트롤러 - 추상화에만 의존하는 구조
# 웹 프레임워크 import, 데이터베이스 코드가 없는 순수한 중재자 역할
from dataclasses import dataclass


@dataclass
class TaskController:
    # 애플리케이션 계층의 유스케이스 인터페이스 (덕 타이핑으로 계약 충족)
    create_use_case: CreateTaskUseCase
    # 인터페이스 계층의 프레젠터 추상화 (ABC로 공식적 인터페이스 계약)
    presenter: TaskPresenter

### 04_web_task_controller_anti_example.py

## 안티 패턴: 웹 프레임워크에 강하게 결합된 컨트롤러

이처럼 신중하게 격리했기 때문에 컨트롤러는 웹 API, 명령 줄 인터페이스(CLI), 메시지 큐 컨슈머 등 어떤 전달 메커니즘에서도 사용할 수 있다. 이 격리를 위반하면 어떤 일이 발생하는지 ﻿살펴보자.

In [ ]:
# 안티 예제: 프레임워크에 강하게 결합된 컨트롤러

try:  # [수정] 안티패턴 예시 - fastapi 설치 불필요
    from fastapi import FastAPI, Request, HTTPException
    from fastapi.responses import JSONResponse
except ImportError:
    # fastapi가 설치되어 있지 않아도 안티패턴 코드 구조를 확인할 수 있도록 스텁 정의
    class FastAPI:  # type: ignore[no-redef]
        pass
    class Request:  # type: ignore[no-redef]
        async def json(self): return {}
    class HTTPException(Exception):  # type: ignore[no-redef]
        def __init__(self, status_code=None, detail=None): pass
    class JSONResponse:  # type: ignore[no-redef]
        def __init__(self, status_code=None, content=None): pass

class ValidationError(Exception):  # [추가] 안티패턴 예제용 스텁
    pass


class WebTaskController:

    def __init__(self, app: FastAPI):

        # 컨트롤러가 FastAPI에 직접 의존
        self.app = app
        # 유스 케이스를 직접 생성하여 강한 결합이 발생
        self.create_use_case = CreateTaskUseCase()

    async def handle_create(self, request: Request):

        try:
            data = await request.json()
            result = self.create_use_case.execute(data)

            return JSONResponse(status_code=201, content={"task": result})

        except ValidationError as e:

            raise HTTPException(status_code=400, detail=str(e))

### 05_operation_result.py

## 오퍼레이션 결과 패턴

`OperationResult`는 컨트롤러 작업의 성공/실패를 명시적으로 표현하는 패턴이다. 인터페이스 어댑터가 오류 사례를 누락하지 않도록 표준화된 결과 전달 구조를 제공한다.

In [ ]:
"""
아키텍처 경계에서는 성공한 작업과 실패한 작업을 모두 처리할 수 있는
명확하고 일관된 방법이 필요하다. 작업은 여러 이유로 실패할 수 있다.
유효하지 않은 입력, 비즈니스 규칙 위반, 시스템 오류 등이 그 원인이며,
각 타입의 실패는 외부 인터페이스에서 서로 다르게 처리해야 할 수 있다.

class TaskController:
    def handle_create(
        self, title: str, description: str
    ) -> OperationResult[TaskViewModel]:
"""

# OperationResult - 컨트롤러 작업의 성공/실패를 명시적으로 표현하는 Either 패턴
# 성공 시 뷰 모델을, 실패 시 에러 뷰 모델을 담아 반환하는 컨테이너
from dataclasses import dataclass
from typing import Generic, TypeVar, Optional  # [보완] Generic, TypeVar, Optional import 추가

T = TypeVar("T")  # [보완] 제네릭 타입 변수 정의


@dataclass
class OperationResult(Generic[T]):
    """컨트롤러 작업의 실행 결과를 표현하는 객체"""

    _success: Optional[T] = None  # 성공 시 뷰 모델 저장

    _error: Optional[ErrorViewModel] = None  # 실패 시 에러 뷰 모델 저장

    # 팩토리 메서드: 성공 결과 생성
    @classmethod
    def succeed(cls, value: T) -> "OperationResult[T]":
        """주어진 뷰 모델을 포함한 성공 결과를 생성"""

        return cls(_success=value)

    # 팩토리 메서드: 실패 결과 생성
    @classmethod
    def fail(cls, message: str, code: Optional[str] = None) -> "OperationResult[T]":
        """오류 정보를 포함한 실패 결과를 생성"""

        return cls(_error=ErrorViewModel(message, code))

### 06_cli_working_with_operation_result.py

## CLI에서 오퍼레이션 결과 활용

CLI 인터페이스에서 오퍼레이션 결과를 처리하는 방법을 보여준다.

In [ ]:
# [수정] OperationResult를 사용하는 CLI 애플리케이션의 의사 코드 예제
# 실제로는 CLI 애플리케이션의 메인 함수 내부에서 실행되는 코드입니다.
# 노트북에서는 의사 코드로만 참고해 주세요.

"""
result = app.task_controller.handle_create(title, description)

if result.is_success:

    task = result.success

    print(f"{task.status_display} [{task.priority_display}] {task.title}")

    return 0

print(result.error.message, fg='red', err=True)

return 1
"""
print("(위 의사 코드는 CLI 애플리케이션 내부 흐름을 보여줍니다)")

### 07_task_controller_data_transform.py

## 컨트롤러의 데이터 변환

컨트롤러는 주입된 의존성에만 의존하며, 유스케이스와 프레젠터 모두 생성자 주입을 통해 전달된다. 아래는 5단계 데이터 변환 흐름이다.

In [ ]:
# TaskController에서의 변환 흐름 예시


def handle_create(self, title: str, description: str) -> OperationResult[TaskViewModel]:

    try:

        # 1. 요청 모델에 대한 외부 입력

        request = CreateTaskRequest(title=title, description=description)

        # 2. 요청 모델을 도메인 실행 작업으로 전달

        result = self.use_case.execute(request)

        if result.is_success:

            # 3. 도메인 실행 결과를 뷰 모델로 변환

            view_model = self.presenter.present_task(result.value)

            return OperationResult.succeed(view_model)

        # 4. 오류 처리 및 형식화

        error_vm = self.presenter.present_error(result.error.message, str(result.error.code.name))

        return OperationResult.fail(error_vm.message, error_vm.code)

    except ValueError as e:

        # 5. 입력 검증 오류 처리

        error_vm = self.presenter.present_error(str(e), "VALIDATION_ERROR")

        return OperationResult.fail(error_vm.message, error_vm.code)


### 08_task_repository.py

## 작업 리포지토리

리포지토리는 컨트롤러/프레젠터와 달리 별도의 어댑터 없이 인터페이스 + 구현만으로 충분하다. 모든 계층 간 상호작용에 어댑터가 필요한 것은 아니다.

> **[추가]** 아래 리포지토리 예제에서 사용하는 `Task` 엔터티를 TodoApp에서 import합니다.

In [ ]:
# [추가] 리포지토리 예제에 필요한 Task 엔터티 import
from todo_app.domain.entities.task import Task

In [ ]:
# 리포지토리 패턴 - 어댑터가 필요 없는 직접 구현 사례
# 컨트롤러/프레젠터와 달리, 리포지토리는 인터페이스 + 구현만으로 충분

# ── 애플리케이션 계층: 리포지토리 인터페이스 정의 (포트) ──
from abc import ABC, abstractmethod
from uuid import UUID


class TaskRepository(ABC):

    @abstractmethod
    def get(self, task_id: UUID) -> Task:
        """ID를 기준으로 작업(Task)을 조회"""

        pass


# ── 인프라 계층: 구체적 구현 (어댑터) ──
# 추상 인터페이스를 직접 구현 - 별도의 컨트롤러/프레젠터 어댑터 불필요

class SqliteTaskRepository(TaskRepository):

    def get(self, task_id: UUID) -> Task:

        # 인터페이스의 구체적 구현 (SQLite 쿼리)

        pass

### 09_display_task.py

## 험블 객체 패턴 (Humble Object Pattern)

모든 형식화 결정(상태, 우선순위, 날짜 표시)은 프레젠터에 존재하고, 뷰는 단순 출력만 수행한다. 이 분리로:
- 뷰가 단순해지고 프레젠테이션 로직이 테스트 가능해짐
- 비즈니스 규칙이 표시 관련 문제로부터 격리됨
- 여러 인터페이스(CLI, 웹 등)가 형식화 로직을 공유 가능

In [ ]:
# 험블 뷰(Humble View) - 로직이 거의 없고 단순하지만 테스트하기는 어려운 뷰
# 모든 형식화 결정은 프레젠터가 담당하고, 뷰는 단순 출력만 수행
# → 프레젠테이션 로직의 테스트 가능성 확보

def display_task(task_vm: TaskViewModel):
    # 뷰 모델의 미리 형식화된 필드를 단순히 출력
    print(f"{task_vm.status_display} [{task_vm.priority_display}] {task_vm.title}")

    if task_vm.due_date_display:

        print(f"Due: {task_vm.due_date_display}")

### 10_task_presenter.py

## 작업 프레젠터

클린 아키텍처의 성공은 아키텍처 경계에서 잘 정의된 인터페이스에 크게 좌우된다. 프레젠터의 경우, 이런 인터페이스는 도메인 데이터를 프레젠테이션에 적합한 형식으로 변환하기 위한 명확한 계약을 수립한다.

작업(Task) 관련 출력을 담당하는 추상 프리젠터

> **[추가]** 아래 프레젠터 예제에서 사용하는 `TaskResponse`, `ErrorViewModel` 등을 TodoApp에서 import합니다.

In [ ]:
# [추가] 프레젠터 예제에 필요한 import
from todo_app.application.dtos.task_dtos import TaskResponse
from todo_app.interfaces.view_models.base import ErrorViewModel
from todo_app.interfaces.view_models.task_vm import TaskViewModel

In [ ]:
# 작업 프레젠터 추상 인터페이스 - 도메인 데이터를 뷰 모델로 변환하는 계약
# 구체 프레젠터(CLI용, 웹용 등)가 이 인터페이스를 구현하여 출력 형식을 결정
from abc import ABC, abstractmethod
from typing import Optional


class TaskPresenter(ABC):
    """작업(Task) 관련 출력을 담당하는 추상 프레젠터"""

    @abstractmethod
    def present_task(self, task_response: TaskResponse) -> TaskViewModel:
        """작업 응답 DTO → 뷰 모델로 변환"""

        pass

    @abstractmethod
    def present_error(self, error_msg: str, code: Optional[str] = None) -> ErrorViewModel:
        """화면 표시를 위한 오류 메시지 형식화"""

        pass

### 11_task_view_model.py

## 작업 뷰 모델

뷰 모델은 프레젠터와 뷰 사이에서 데이터를 전달하는 역할을 하여, 프레젠테이션 로직과 표시 관련 사항을 깔끔하게 분리한다. 뷰 모델은 어떤 뷰 구현에서도 쉽게 사용할 수 있도록 형식화된 데이터를 캡슐화한다.

화면(UI) 표시를 위한 작업 전용 표현 모델

In [ ]:
# 작업 뷰 모델 - 프레젠터와 뷰 사이의 데이터 전달 객체
# 불변(frozen) 데이터클래스로, 미리 형식화된 문자열 필드만 포함
# → 뷰가 추가 로직 없이 단순 출력만 수행하도록 보장
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class TaskViewModel:
    """화면(UI) 표시를 위한 작업 전용 표현 모델"""

    id: str
    title: str
    description: str
    status_display: str  # 프레젠터가 미리 형식화한 상태 문자열
    priority_display: str  # 프레젠터가 미리 형식화한 우선순위 문자열
    due_date_display: Optional[str]  # 프레젠터가 미리 형식화한 마감일 문자열
    project_display: Optional[str]  # 프레젠터가 미리 형식화한 프로젝트 정보
    completion_info: Optional[str]  # 프레젠터가 미리 형식화한 완료 정보

### 12_cli_task_presenter.py

## CLI 작업 프레젠터

프레젠터 인터페이스와 뷰 모델을 정의했으므로 특정 인터페이스 요구 사항에 맞는 구체적인 프레젠터를 구현할 수 있을 것이다. 구체적인 프레젠터는 프레임워크 및 드라이버 계층에서 구현되지만, 맥락을 위해 여기서 미리 ﻿살펴본다.

CLI 환경에 특화된 작업(Task) 프리젠터

In [ ]:
# CLI 작업 프레젠터 - TaskPresenter 추상 인터페이스의 구체적 구현
# 터미널 출력에 최적화된 형식으로 도메인 데이터를 변환하는 어댑터
from datetime import datetime, timezone
from typing import Optional


class CliTaskPresenter(TaskPresenter):
    """CLI 환경에 특화된 작업(Task) 프레젠터"""

    # 도메인 응답 DTO → CLI용 뷰 모델로 변환하는 핵심 메서드
    def present_task(self, task_response: TaskResponse) -> TaskViewModel:
        """작업 정보를 CLI 출력용 형식으로 변환"""

        return TaskViewModel(
            id=str(task_response.id),
            title=task_response.title,
            description=task_response.description,
            status_display=self._format_status(task_response.status),
            priority_display=self._format_priority(task_response.priority),
            due_date_display=self._format_due_date(task_response.due_date),
            project_display=self._format_project(task_response.project_id),
            completion_info=self._format_completion_info(
                task_response.completion_date, task_response.completion_notes
            ),
        )

    # 헬퍼 메서드: 상태 열거형 → CLI 표시 문자열로 변환
    def _format_status(self, status) -> str:  # [보완] 누락된 헬퍼 메서드 추가
        """상태를 CLI 표시용으로 형식화"""
        return f"[{status.value}]"

    # 헬퍼 메서드: 우선순위 열거형 → CLI 표시 문자열로 변환
    def _format_priority(self, priority) -> str:  # [보완] 누락된 헬퍼 메서드 추가
        """CLI 표시를 위한 우선순위 형식화"""
        from todo_app.domain.value_objects import Priority
        display_map = {
            Priority.LOW: "Minor",
            Priority.MEDIUM: "Normal",
            Priority.HIGH: "High",
        }
        return display_map.get(priority, str(priority))

    # 헬퍼 메서드: 마감일 형식화 + 초과 여부 표시
    def _format_due_date(self, due_date: Optional[datetime]) -> str:
        """마감일을 포맷하고, 기한 초과 여부를 표시"""

        if not due_date:

            return "No due date"

        is_overdue = due_date < datetime.now(timezone.utc)

        date_str = due_date.strftime("%Y-%m-%d")

        return f"OVERDUE - Due: {date_str}" if is_overdue else f"Due: {date_str}"

    def _format_project(self, project_id) -> str:  # [보완] 누락된 헬퍼 메서드 추가
        """프로젝트 정보를 CLI 표시용으로 형식화"""
        return f"Project: {project_id}" if project_id else ""

    def _format_completion_info(self, completion_date, completion_notes) -> str:  # [보완] 누락된 헬퍼 메서드 추가
        """완료 정보를 형식화하고, 메모가 있으면 포함"""
        if not completion_date:
            return "Not completed"
        base_info = f"Completed on {completion_date.strftime('%Y-%m-%d %H:%M')}"
        if completion_notes:
            return f"{base_info} - {completion_notes}"
        return base_info

    # 에러 뷰 모델 생성
    def present_error(self, error_msg: str, code: Optional[str] = None) -> ErrorViewModel:
        """CLI 출력용 오류 메시지 형식화"""

        return ErrorViewModel(message=error_msg, code=code)